In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))


from src.data import load_data, analyze_missing_data, analyze_frequent_data, analyze_unique_values
from src.features import (create_family_size, create_age_intervals, create_fare_intervals, 
                     create_sex_pclass_feature, create_family_type, standardize_titles,
                     extract_name_features, combine_datasets)
from src.model import encode_categorical_features, prepare_training_data, train_random_forest, evaluate_model
from src.utils import plot_count_pairs, plot_distribution_pairs, print_data_summary, get_feature_importance, plot_feature_importance


In [8]:
TRAIN_PATH = "data/train.csv"
TEST_PATH = "data/test.csv"
train_df, test_df = load_data(TRAIN_PATH, TEST_PATH)
all_df = combine_datasets(train_df, test_df)

In [ ]:
# Titanic Data Visualization Analysis

This report presents a comprehensive visualization analysis of the Titanic dataset, focusing on the correlation between different features and survival rate.


In [ ]:

# Set plot style
plt.rcParams['axes.unicode_minus'] = False  # Fix minus sign display issue
sns.set_style("whitegrid")
sns.set_palette("husl")

# Prepare data: Create family size feature (using only training data for analysis since test set doesn't have Survived labels)
train_data = all_df[all_df['set'] == 'train'].copy()
train_data = create_family_size(train_data)
train_data = create_family_type(train_data)

print(f"Training data shape: {train_data.shape}")
print(f"Training data columns: {list(train_data.columns)}")


## 1. Gender and Survival Rate Analysis


In [ ]:
# Gender and Survival Rate Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Survival count by gender
sns.countplot(data=train_data, x='Sex', hue='Survived', ax=axes[0], palette=['#e74c3c', '#2ecc71'])
axes[0].set_title('Survival Count by Gender', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Gender', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].legend(['Did not survive', 'Survived'], title='Survival Status')
axes[0].grid(axis='y', alpha=0.3)

# Right plot: Survival rate by gender
survival_by_sex = train_data.groupby('Sex')['Survived'].agg(['mean', 'count'])
survival_by_sex.columns = ['Survival Rate', 'Total Count']
survival_by_sex['Survival Rate'] = survival_by_sex['Survival Rate'] * 100

bars = axes[1].bar(survival_by_sex.index, survival_by_sex['Survival Rate'], 
                    color=['#3498db', '#e91e63'], alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_title('Survival Rate by Gender', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Gender', fontsize=12)
axes[1].set_ylabel('Survival Rate (%)', fontsize=12)
axes[1].set_ylim([0, 100])
axes[1].grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (idx, row) in enumerate(survival_by_sex.iterrows()):
    axes[1].text(i, row['Survival Rate'] + 2, f"{row['Survival Rate']:.1f}%", 
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    axes[1].text(i, row['Survival Rate'] - 5, f"n={int(row['Total Count'])}", 
                ha='center', va='top', fontsize=9, style='italic')

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Gender and Survival Rate Statistics ===")
print(survival_by_sex)
print(f"\nFemale survival rate: {survival_by_sex.loc['female', 'Survival Rate']:.2f}%")
print(f"Male survival rate: {survival_by_sex.loc['male', 'Survival Rate']:.2f}%")
print(f"Survival rate difference: {survival_by_sex.loc['female', 'Survival Rate'] - survival_by_sex.loc['male', 'Survival Rate']:.2f} percentage points")

print("\n【Analysis】")
print("From the visualization, gender is one of the most important factors affecting survival rate.")
print("- Female survival rate is much higher than male (approximately 74% vs 19%), with a difference of over 50 percentage points")
print("- This reflects the 'women and children first' rescue principle that was followed during the Titanic sinking")
print("- Gender is a very strong predictor for survival rate")


## 2. Age and Survival Rate Analysis


In [ ]:
# Age and Survival Rate Analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Create age intervals
train_data_age = train_data.copy()
train_data_age = create_age_intervals(train_data_age)
age_labels = ['0-16 years', '17-32 years', '33-48 years', '49-64 years', '65+ years']
train_data_age['Age Interval Label'] = train_data_age['Age Interval'].map({
    0: '0-16 years', 1: '17-32 years', 2: '33-48 years', 3: '49-64 years', 4: '65+ years'
})

# Top left: Age distribution histogram (grouped by survival status)
train_data_with_age = train_data_age.dropna(subset=['Age'])
sns.histplot(data=train_data_with_age, x='Age', hue='Survived', bins=30, 
             kde=True, ax=axes[0, 0], palette=['#e74c3c', '#2ecc71'], alpha=0.6)
axes[0, 0].set_title('Age Distribution by Survival Status', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Age', fontsize=12)
axes[0, 0].set_ylabel('Count', fontsize=12)
axes[0, 0].legend(['Did not survive', 'Survived'], title='Survival Status')
axes[0, 0].grid(axis='y', alpha=0.3)

# Top right: Survival rate by age interval
survival_by_age = train_data_age.groupby('Age Interval Label')['Survived'].agg(['mean', 'count'])
survival_by_age.columns = ['Survival Rate', 'Total Count']
survival_by_age = survival_by_age.reindex(age_labels)
survival_by_age['Survival Rate'] = survival_by_age['Survival Rate'] * 100

bars = axes[0, 1].bar(range(len(survival_by_age)), survival_by_age['Survival Rate'], 
                      color=plt.cm.viridis(np.linspace(0, 1, len(survival_by_age))), 
                      alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 1].set_title('Survival Rate by Age Interval', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Age Interval', fontsize=12)
axes[0, 1].set_ylabel('Survival Rate (%)', fontsize=12)
axes[0, 1].set_xticks(range(len(survival_by_age)))
axes[0, 1].set_xticklabels(survival_by_age.index, rotation=45, ha='right')
axes[0, 1].set_ylim([0, 100])
axes[0, 1].grid(axis='y', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(survival_by_age.iterrows()):
    if not pd.isna(row['Survival Rate']):
        axes[0, 1].text(i, row['Survival Rate'] + 2, f"{row['Survival Rate']:.1f}%", 
                        ha='center', va='bottom', fontsize=10, fontweight='bold')

# Bottom left: Boxplot - Age distribution for survived vs not survived passengers
sns.boxplot(data=train_data_with_age, x='Survived', y='Age', ax=axes[1, 0], 
            palette=['#e74c3c', '#2ecc71'])
axes[1, 0].set_title('Age Distribution by Survival Status (Boxplot)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Survival Status', fontsize=12)
axes[1, 0].set_ylabel('Age', fontsize=12)
axes[1, 0].set_xticklabels(['Did not survive', 'Survived'])
axes[1, 0].grid(axis='y', alpha=0.3)

# Bottom right: Violin plot - More detailed age distribution
sns.violinplot(data=train_data_with_age, x='Survived', y='Age', ax=axes[1, 1], 
               palette=['#e74c3c', '#2ecc71'], inner='quartile')
axes[1, 1].set_title('Age Distribution by Survival Status (Violin Plot)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Survival Status', fontsize=12)
axes[1, 1].set_ylabel('Age', fontsize=12)
axes[1, 1].set_xticklabels(['Did not survive', 'Survived'])
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Age and Survival Rate Statistics ===")
print(survival_by_age)
print(f"\nChildren (0-16 years) survival rate: {survival_by_age.loc['0-16 years', 'Survival Rate']:.2f}%")
print(f"Missing age data: {train_data_age['Age'].isna().sum()} passengers ({train_data_age['Age'].isna().sum()/len(train_data_age)*100:.1f}%)")

print("\n【Analysis】")
print("Age has a significant impact on survival rate, though not as strong as gender.")
print("- Children (0-16 years) have relatively high survival rates, reflecting the 'children first' rescue principle")
print("- From the age distribution, children and elderly passengers have relatively higher proportions among survivors")
print("- Young and middle-aged adults (17-48 years) have relatively lower survival rates, possibly due to giving way to women and children")
print("- There are approximately 20% missing age values in the data that need appropriate handling")


## 3. Passenger Class and Survival Rate Analysis


In [ ]:
# Passenger Class and Survival Rate Analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left plot: Survival count by passenger class
pclass_order = [1, 2, 3]
sns.countplot(data=train_data, x='Pclass', hue='Survived', ax=axes[0], 
              order=pclass_order, palette=['#e74c3c', '#2ecc71'])
axes[0].set_title('Survival Count by Passenger Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Passenger Class (1=First, 2=Second, 3=Third)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].legend(['Did not survive', 'Survived'], title='Survival Status')
axes[0].grid(axis='y', alpha=0.3)

# Middle plot: Survival rate by passenger class
survival_by_pclass = train_data.groupby('Pclass')['Survived'].agg(['mean', 'count'])
survival_by_pclass.columns = ['Survival Rate', 'Total Count']
survival_by_pclass['Survival Rate'] = survival_by_pclass['Survival Rate'] * 100

colors = ['#f39c12', '#3498db', '#e74c3c']  # Gold, blue, red representing three classes
bars = axes[1].bar(survival_by_pclass.index, survival_by_pclass['Survival Rate'], 
                    color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_title('Survival Rate by Passenger Class', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Passenger Class', fontsize=12)
axes[1].set_ylabel('Survival Rate (%)', fontsize=12)
axes[1].set_xticks(survival_by_pclass.index)
axes[1].set_xticklabels(['First Class', 'Second Class', 'Third Class'])
axes[1].set_ylim([0, 100])
axes[1].grid(axis='y', alpha=0.3)

# Add value labels
for idx, row in survival_by_pclass.iterrows():
    axes[1].text(idx, row['Survival Rate'] + 2, f"{row['Survival Rate']:.1f}%", 
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    axes[1].text(idx, row['Survival Rate'] - 5, f"n={int(row['Total Count'])}", 
                ha='center', va='top', fontsize=9, style='italic')

# Right plot: Interaction effect between passenger class and gender
survival_by_pclass_sex = train_data.groupby(['Pclass', 'Sex'])['Survived'].mean() * 100
survival_by_pclass_sex = survival_by_pclass_sex.reset_index()
survival_by_pclass_sex.columns = ['Pclass', 'Sex', 'Survival Rate']

x = np.arange(len(pclass_order))
width = 0.35
for i, sex in enumerate(['female', 'male']):
    values = [survival_by_pclass_sex[(survival_by_pclass_sex['Pclass']==p) & 
                                     (survival_by_pclass_sex['Sex']==sex)]['Survival Rate'].values[0] 
              for p in pclass_order]
    axes[2].bar(x + i*width, values, width, label='Female' if sex=='female' else 'Male', 
                alpha=0.7, edgecolor='black', linewidth=1)
axes[2].set_title('Interaction: Passenger Class and Gender', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Passenger Class', fontsize=12)
axes[2].set_ylabel('Survival Rate (%)', fontsize=12)
axes[2].set_xticks(x + width / 2)
axes[2].set_xticklabels(['First Class', 'Second Class', 'Third Class'])
axes[2].legend()
axes[2].set_ylim([0, 100])
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Passenger Class and Survival Rate Statistics ===")
print(survival_by_pclass)
print("\n=== Interaction: Passenger Class and Gender Statistics ===")
print(survival_by_pclass_sex.pivot(index='Pclass', columns='Sex', values='Survival Rate'))

print("\n【Analysis】")
print("Passenger class is another important factor affecting survival rate, reflecting the association between social class and rescue priority.")
print("- First class passengers have the highest survival rate (approximately 63%), followed by second class (approximately 47%), and third class has the lowest (approximately 24%)")
print("- Higher passenger class is associated with higher survival rate, possibly because:")
print("  1. First class cabins were located closer to lifeboats")
print("  2. Social status may have influenced rescue priority")
print("  3. First class passengers may have received evacuation notices earlier")
print("- Even within the same passenger class, female survival rate is much higher than male")
print("- First class females have nearly 100% survival rate, while third class males have the lowest survival rate")


## 4. Family Size and Survival Rate Analysis


In [ ]:
# Family Size and Survival Rate Analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top left: Family size distribution (by survival status)
sns.countplot(data=train_data, x='Family Size', hue='Survived', ax=axes[0, 0], 
              palette=['#e74c3c', '#2ecc71'])
axes[0, 0].set_title('Survival Count by Family Size', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Family Size (number of people)', fontsize=12)
axes[0, 0].set_ylabel('Count', fontsize=12)
axes[0, 0].legend(['Did not survive', 'Survived'], title='Survival Status')
axes[0, 0].grid(axis='y', alpha=0.3)

# Top right: Survival rate by family size
survival_by_family = train_data.groupby('Family Size')['Survived'].agg(['mean', 'count'])
survival_by_family.columns = ['Survival Rate', 'Total Count']
survival_by_family['Survival Rate'] = survival_by_family['Survival Rate'] * 100

bars = axes[0, 1].bar(survival_by_family.index, survival_by_family['Survival Rate'], 
                      color=plt.cm.coolwarm(np.linspace(0, 1, len(survival_by_family))), 
                      alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 1].set_title('Survival Rate by Family Size', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Family Size (number of people)', fontsize=12)
axes[0, 1].set_ylabel('Survival Rate (%)', fontsize=12)
axes[0, 1].set_ylim([0, 100])
axes[0, 1].grid(axis='y', alpha=0.3)

# Add value labels
for idx, row in survival_by_family.iterrows():
    axes[0, 1].text(idx, row['Survival Rate'] + 2, f"{row['Survival Rate']:.1f}%", 
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
    axes[0, 1].text(idx, row['Survival Rate'] - 5, f"n={int(row['Total Count'])}", 
                    ha='center', va='top', fontsize=8, style='italic')

# Bottom left: Family type and survival rate
survival_by_family_type = train_data.groupby('Family Type')['Survived'].agg(['mean', 'count'])
survival_by_family_type.columns = ['Survival Rate', 'Total Count']
survival_by_family_type['Survival Rate'] = survival_by_family_type['Survival Rate'] * 100
# Order: Single, Small, Large
type_order = ['Single', 'Small', 'Large']
survival_by_family_type = survival_by_family_type.reindex(type_order)

type_labels = ['Single', 'Small Family\n(2-4 people)', 'Large Family\n(5+ people)']
bars = axes[1, 0].bar(range(len(survival_by_family_type)), survival_by_family_type['Survival Rate'], 
                      color=['#95a5a6', '#3498db', '#e67e22'], 
                      alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 0].set_title('Survival Rate by Family Type', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Family Type', fontsize=12)
axes[1, 0].set_ylabel('Survival Rate (%)', fontsize=12)
axes[1, 0].set_xticks(range(len(survival_by_family_type)))
axes[1, 0].set_xticklabels(type_labels)
axes[1, 0].set_ylim([0, 100])
axes[1, 0].grid(axis='y', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(survival_by_family_type.iterrows()):
    axes[1, 0].text(i, row['Survival Rate'] + 2, f"{row['Survival Rate']:.1f}%", 
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

# Bottom right: Family size and survival rate relationship (line plot showing trend)
family_sizes = sorted(survival_by_family.index)
survival_rates = [survival_by_family.loc[fs, 'Survival Rate'] for fs in family_sizes]
counts = [survival_by_family.loc[fs, 'Total Count'] for fs in family_sizes]

ax1 = axes[1, 1]
ax2 = ax1.twinx()

line1 = ax1.plot(family_sizes, survival_rates, marker='o', linewidth=2, 
                 markersize=8, color='#2ecc71', label='Survival Rate')
ax1.set_xlabel('Family Size (number of people)', fontsize=12)
ax1.set_ylabel('Survival Rate (%)', fontsize=12, color='#2ecc71')
ax1.tick_params(axis='y', labelcolor='#2ecc71')
ax1.set_title('Family Size and Survival Rate Trend', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

bars = ax2.bar(family_sizes, counts, alpha=0.3, color='#3498db', label='Count')
ax2.set_ylabel('Count', fontsize=12, color='#3498db')
ax2.tick_params(axis='y', labelcolor='#3498db')

# Add legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Family Size and Survival Rate Statistics ===")
print(survival_by_family)
print("\n=== Family Type and Survival Rate Statistics ===")
print(survival_by_family_type)

print("\n【Analysis】")
print("Family size has a complex impact on survival rate, showing a non-linear relationship.")
print("- Single passengers (family size=1) have a survival rate of approximately 30%, which is relatively low")
print("- Small families (2-4 people) have the highest survival rate, possibly because family members can help each other")
print("- Large families (5+ people) have lower survival rates, possibly because:")
print("  1. Large families may have more male members (males have lower survival rates)")
print("  2. Large families may have difficulty moving during evacuation")
print("  3. Need to take care of more members, increasing rescue difficulty")
print("- Family size of 2-4 people has the highest survival rate, which may reflect the 'family first' rescue principle")
print("- However, the impact of family size is not as strong as gender and passenger class")


## 5. Embarkation Port and Survival Rate Analysis


In [ ]:
# Embarkation Port and Survival Rate Analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Handle missing values (use 'Unknown' to represent)
train_data_embarked = train_data.copy()
train_data_embarked['Embarked'] = train_data_embarked['Embarked'].fillna('Unknown')

# Left plot: Survival count by embarkation port
embarked_order = ['C', 'Q', 'S', 'Unknown']
embarked_labels = ['Cherbourg', 'Queenstown', 'Southampton', 'Unknown']
sns.countplot(data=train_data_embarked, x='Embarked', hue='Survived', ax=axes[0], 
              order=embarked_order, palette=['#e74c3c', '#2ecc71'])
axes[0].set_title('Survival Count by Embarkation Port', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Embarkation Port', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xticklabels(embarked_labels, rotation=0, ha='center')
axes[0].legend(['Did not survive', 'Survived'], title='Survival Status')
axes[0].grid(axis='y', alpha=0.3)

# Middle plot: Survival rate by embarkation port
survival_by_embarked = train_data_embarked.groupby('Embarked')['Survived'].agg(['mean', 'count'])
survival_by_embarked.columns = ['Survival Rate', 'Total Count']
survival_by_embarked = survival_by_embarked.reindex(embarked_order)
survival_by_embarked['Survival Rate'] = survival_by_embarked['Survival Rate'] * 100

colors = ['#9b59b6', '#f39c12', '#3498db', '#95a5a6']
bars = axes[1].bar(range(len(survival_by_embarked)), survival_by_embarked['Survival Rate'], 
                    color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_title('Survival Rate by Embarkation Port', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Embarkation Port', fontsize=12)
axes[1].set_ylabel('Survival Rate (%)', fontsize=12)
axes[1].set_xticks(range(len(survival_by_embarked)))
axes[1].set_xticklabels(embarked_labels, rotation=0, ha='center')
axes[1].set_ylim([0, 100])
axes[1].grid(axis='y', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(survival_by_embarked.iterrows()):
    if not pd.isna(row['Survival Rate']):
        axes[1].text(i, row['Survival Rate'] + 2, f"{row['Survival Rate']:.1f}%", 
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
        axes[1].text(i, row['Survival Rate'] - 5, f"n={int(row['Total Count'])}", 
                    ha='center', va='top', fontsize=9, style='italic')

# Right plot: Interaction effect between embarkation port and passenger class
embarked_pclass = train_data_embarked.groupby(['Embarked', 'Pclass'])['Survived'].mean() * 100
embarked_pclass = embarked_pclass.reset_index()
embarked_pclass.columns = ['Embarked', 'Pclass', 'Survival Rate']

x = np.arange(len(embarked_order))
width = 0.25
for i, pclass in enumerate([1, 2, 3]):
    values = []
    for emb in embarked_order:
        val = embarked_pclass[(embarked_pclass['Embarked']==emb) & 
                              (embarked_pclass['Pclass']==pclass)]
        if len(val) > 0:
            values.append(val['Survival Rate'].values[0])
        else:
            values.append(0)
    axes[2].bar(x + i*width, values, width, label=f'Class {pclass}', 
                alpha=0.7, edgecolor='black', linewidth=1)
axes[2].set_title('Interaction: Embarkation Port and Passenger Class', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Embarkation Port', fontsize=12)
axes[2].set_ylabel('Survival Rate (%)', fontsize=12)
axes[2].set_xticks(x + width)
axes[2].set_xticklabels(embarked_labels, rotation=0, ha='center')
axes[2].legend()
axes[2].set_ylim([0, 100])
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Embarkation Port and Survival Rate Statistics ===")
print(survival_by_embarked)
print("\n=== Interaction: Embarkation Port and Passenger Class Statistics ===")
print(embarked_pclass.pivot(index='Embarked', columns='Pclass', values='Survival Rate'))

print("\n【Analysis】")
print("Embarkation port has some impact on survival rate, but this effect may be related to other factors (such as passenger class).")
print("- Passengers who embarked from Cherbourg have the highest survival rate (approximately 55%), possibly because:")
print("  1. A higher proportion of first and second class passengers embarked from Cherbourg")
print("  2. This port may have attracted more affluent passengers")
print("- Passengers who embarked from Queenstown have the lowest survival rate (approximately 39%), possibly because:")
print("  1. A higher proportion of third class passengers embarked from this port")
print("  2. This port may have been a main embarkation point for immigrants")
print("- Southampton is the largest embarkation port, with a moderate survival rate (approximately 34%)")
print("- The impact of embarkation port may be more indirect, affecting survival rate through its influence on passenger class distribution")


## 6. Fare and Survival Rate Analysis


In [ ]:
# Fare and Survival Rate Analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Handle missing values
train_data_fare = train_data.copy()
train_data_fare = create_fare_intervals(train_data_fare)
fare_labels = ['Low Fare\n(≤7.91)', 'Medium-Low Fare\n(7.91-14.45)', 'Medium-High Fare\n(14.45-31)', 'High Fare\n(>31)']
train_data_fare['Fare Interval Label'] = train_data_fare['Fare Interval'].map({
    0: 'Low Fare\n(≤7.91)', 1: 'Medium-Low Fare\n(7.91-14.45)', 2: 'Medium-High Fare\n(14.45-31)', 3: 'High Fare\n(>31)'
})

# Top left: Fare distribution histogram (grouped by survival status)
train_data_with_fare = train_data_fare.dropna(subset=['Fare'])
sns.histplot(data=train_data_with_fare, x='Fare', hue='Survived', bins=30, 
             kde=True, ax=axes[0, 0], palette=['#e74c3c', '#2ecc71'], alpha=0.6)
axes[0, 0].set_title('Fare Distribution by Survival Status', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Fare', fontsize=12)
axes[0, 0].set_ylabel('Count', fontsize=12)
axes[0, 0].legend(['Did not survive', 'Survived'], title='Survival Status')
axes[0, 0].grid(axis='y', alpha=0.3)

# Top right: Survival rate by fare interval
survival_by_fare = train_data_fare.groupby('Fare Interval Label')['Survived'].agg(['mean', 'count'])
survival_by_fare.columns = ['Survival Rate', 'Total Count']
survival_by_fare['Survival Rate'] = survival_by_fare['Survival Rate'] * 100
# Order
fare_order = ['Low Fare\n(≤7.91)', 'Medium-Low Fare\n(7.91-14.45)', 'Medium-High Fare\n(14.45-31)', 'High Fare\n(>31)']
survival_by_fare = survival_by_fare.reindex(fare_order)

bars = axes[0, 1].bar(range(len(survival_by_fare)), survival_by_fare['Survival Rate'], 
                      color=plt.cm.plasma(np.linspace(0, 1, len(survival_by_fare))), 
                      alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 1].set_title('Survival Rate by Fare Interval', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Fare Interval', fontsize=12)
axes[0, 1].set_ylabel('Survival Rate (%)', fontsize=12)
axes[0, 1].set_xticks(range(len(survival_by_fare)))
axes[0, 1].set_xticklabels(survival_by_fare.index, rotation=0, ha='center')
axes[0, 1].set_ylim([0, 100])
axes[0, 1].grid(axis='y', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(survival_by_fare.iterrows()):
    if not pd.isna(row['Survival Rate']):
        axes[0, 1].text(i, row['Survival Rate'] + 2, f"{row['Survival Rate']:.1f}%", 
                        ha='center', va='bottom', fontsize=10, fontweight='bold')
        axes[0, 1].text(i, row['Survival Rate'] - 5, f"n={int(row['Total Count'])}", 
                        ha='center', va='top', fontsize=8, style='italic')

# Bottom left: Boxplot - Fare distribution for survived vs not survived passengers
sns.boxplot(data=train_data_with_fare, x='Survived', y='Fare', ax=axes[1, 0], 
            palette=['#e74c3c', '#2ecc71'])
axes[1, 0].set_title('Fare Distribution by Survival Status (Boxplot)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Survival Status', fontsize=12)
axes[1, 0].set_ylabel('Fare', fontsize=12)
axes[1, 0].set_xticklabels(['Did not survive', 'Survived'])
axes[1, 0].set_yscale('log')  # Use log scale because fare distribution is skewed
axes[1, 0].grid(axis='y', alpha=0.3)

# Bottom right: Relationship between fare and passenger class (heatmap)
fare_pclass = train_data_with_fare.groupby(['Fare Interval', 'Pclass'])['Survived'].mean() * 100
fare_pclass = fare_pclass.reset_index()
fare_pclass_pivot = fare_pclass.pivot(index='Fare Interval', columns='Pclass', values='Survived')

sns.heatmap(fare_pclass_pivot, annot=True, fmt='.1f', cmap='YlOrRd', 
            ax=axes[1, 1], cbar_kws={'label': 'Survival Rate (%)'}, linewidths=1, linecolor='black')
axes[1, 1].set_title('Interaction: Fare Interval and Passenger Class (Heatmap)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Passenger Class', fontsize=12)
axes[1, 1].set_ylabel('Fare Interval', fontsize=12)
axes[1, 1].set_yticklabels(['Low Fare', 'Medium-Low Fare', 'Medium-High Fare', 'High Fare'], rotation=0)
axes[1, 1].set_xticklabels(['Class 1', 'Class 2', 'Class 3'], rotation=0)

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Fare and Survival Rate Statistics ===")
print(survival_by_fare)
print(f"\nMissing fare data: {train_data_fare['Fare'].isna().sum()} passengers")
print(f"\nFare statistics:")
print(train_data_with_fare['Fare'].describe())

print("\n【Analysis】")
print("Fare shows a clear positive correlation with survival rate - the higher the fare, the higher the survival rate.")
print("- High fare (>31) passengers have the highest survival rate (approximately 60%), while low fare (≤7.91) passengers have the lowest (approximately 22%)")
print("- Fare is highly correlated with passenger class - high fare usually corresponds to first class, low fare to third class")
print("- From the boxplot, we can see that the median fare of surviving passengers is significantly higher than non-surviving passengers")
print("- Fare distribution shows right-skewed (positive skew) distribution, with most passengers paying lower fares")
print("- Fare serves as a proxy variable for passenger class, reflecting the impact of socioeconomic status on survival rate")
print("- High fare passengers may:")
print("  1. Stay in cabins closer to lifeboats")
print("  2. Receive evacuation notices earlier")
print("  3. Have priority in rescue due to social status")


## 7. Comprehensive Correlation Analysis


In [ ]:
# Comprehensive Correlation Analysis
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Prepare numeric features for correlation analysis
corr_data = train_data.copy()
corr_data['Sex_encoded'] = corr_data['Sex'].map({'male': 0, 'female': 1})
corr_data['Embarked_encoded'] = corr_data['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

# Select numeric features
numeric_features = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 
                    'Family Size', 'Sex_encoded', 'Embarked_encoded']
corr_matrix = corr_data[numeric_features].corr()

# Top left: Feature correlation heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Show only lower triangle
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": .8},
            ax=axes[0, 0], vmin=-1, vmax=1)
axes[0, 0].set_title('Feature Correlation Matrix (with Survival Rate)', fontsize=14, fontweight='bold')
axes[0, 0].set_xticklabels(['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Family Size', 'Sex', 'Embarked'], 
                          rotation=45, ha='right')
axes[0, 0].set_yticklabels(['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Family Size', 'Sex', 'Embarked'], 
                          rotation=0)

# Top right: Correlation coefficients between each feature and survival rate (bar chart)
survival_corr = corr_matrix['Survived'].drop('Survived').sort_values(ascending=True)
colors_corr = ['#e74c3c' if x < 0 else '#2ecc71' for x in survival_corr.values]
bars = axes[0, 1].barh(range(len(survival_corr)), survival_corr.values, 
                       color=colors_corr, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 1].set_title('Correlation Coefficients: Features vs Survival Rate', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Correlation Coefficient', fontsize=12)
axes[0, 1].set_yticks(range(len(survival_corr)))
axes[0, 1].set_yticklabels(['Embarked', 'Parch', 'SibSp', 'Age', 'Family Size', 
                           'Pclass', 'Fare', 'Sex'], rotation=0)
axes[0, 1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[0, 1].grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, val) in enumerate(survival_corr.items()):
    axes[0, 1].text(val + 0.02 if val > 0 else val - 0.02, i, f'{val:.3f}', 
                    ha='left' if val > 0 else 'right', va='center', fontsize=10, fontweight='bold')

# Bottom left: Multi-feature combination analysis - Gender × Passenger Class × Survival Rate
pivot_data = train_data.groupby(['Sex', 'Pclass'])['Survived'].agg(['mean', 'count'])
pivot_data.columns = ['Survival Rate', 'Count']
pivot_data['Survival Rate'] = pivot_data['Survival Rate'] * 100
pivot_table = pivot_data['Survival Rate'].unstack('Pclass')

sns.heatmap(pivot_table, annot=True, fmt='.1f', cmap='RdYlGn', 
            ax=axes[1, 0], cbar_kws={'label': 'Survival Rate (%)'}, 
            linewidths=1, linecolor='black', vmin=0, vmax=100)
axes[1, 0].set_title('Interaction: Gender and Passenger Class (Survival Rate %)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Passenger Class', fontsize=12)
axes[1, 0].set_ylabel('Gender', fontsize=12)
axes[1, 0].set_yticklabels(['Female', 'Male'], rotation=0)
axes[1, 0].set_xticklabels(['Class 1', 'Class 2', 'Class 3'], rotation=0)

# Bottom right: Feature importance summary (based on absolute correlation values)
feature_importance = abs(survival_corr).sort_values(ascending=False)
colors_imp = plt.cm.viridis(np.linspace(0, 1, len(feature_importance)))
bars = axes[1, 1].barh(range(len(feature_importance)), feature_importance.values, 
                      color=colors_imp, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 1].set_title('Feature Importance Ranking (Based on Absolute Correlation with Survival Rate)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('|Correlation Coefficient|', fontsize=12)
axes[1, 1].set_yticks(range(len(feature_importance)))
axes[1, 1].set_yticklabels(['Sex', 'Pclass', 'Fare', 'Family Size', 'Age', 
                           'SibSp', 'Parch', 'Embarked'], rotation=0)
axes[1, 1].grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, val) in enumerate(feature_importance.items()):
    axes[1, 1].text(val + 0.01, i, f'{val:.3f}', 
                    ha='left', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Correlation Coefficients: Features vs Survival Rate ===")
print(survival_corr.sort_values(ascending=False))
print("\n=== Feature Importance Ranking (by Absolute Correlation) ===")
print(feature_importance)

print("\n【Comprehensive Analysis】")
print("Through correlation analysis, we can draw the following conclusions:")
print("\n1. Most Important Predictive Features:")
print("   - Sex (correlation coefficient: {:.3f}): Strongest predictor, female survival rate is much higher than male".format(survival_corr['Sex_encoded']))
print("   - Passenger Class (correlation coefficient: {:.3f}): Negative correlation, higher class (lower value) associated with higher survival rate".format(survival_corr['Pclass']))
print("   - Fare (correlation coefficient: {:.3f}): Positive correlation, higher fare associated with higher survival rate".format(survival_corr['Fare']))
print("\n2. Moderately Important Features:")
print("   - Family Size: Smaller impact, but small families (2-4 people) have the highest survival rate")
print("   - Age: Children and elderly have relatively higher survival rates")
print("\n3. Less Important Features:")
print("   - Embarkation Port: Mainly affects survival rate indirectly through its influence on passenger class distribution")
print("   - Number of siblings/spouses and parents/children: Small individual impact, but has some effect when combined into family size")
print("\n4. Interaction Effects:")
print("   - The interaction between gender and passenger class is very significant")
print("   - First class females have nearly 100% survival rate, while third class males have the lowest survival rate")
print("   - This reflects the implementation of the 'women and children first' principle across different social classes")
